<a href="https://colab.research.google.com/github/sanjana19991118/engineering-mindmaps-Knowledge-base/blob/main/Advanced_Apex_Project_2_Week2_Sanjana_V_12000191_skin_cancer_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Apex Project II (M.Sc. DS & AI)


---



Name : Sanjana V

Student Id : 1200019

email : 2025em1200019@bitspilani-digital.edu.in



| Attribute | Details |
| :--- | :--- |
| **Selected problem** | P5 - CNN-Based Medical Image Classification for Disease Detection |
| **Dataset** | HAM10000 Skin Cancer MNIST Dataset (Kaggle) |
| **Task type** | Multi-class image classification |
| **Main model** | Convolutional Neural Network (CNN) / Transfer Learning (VGG16, ResNet50) for comparison |
| **Target** | `dx` --> Diagnosis (The 7-class lesion category being predicted) |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

 # Step 1: Problem Statement & Business Goal

**Problem Statement**


---


**CNN-Based Medical Image Classification for Disease Detection**

## Dataset: HAM10000 Skin Cancer MNIST Dataset
## **Dataset Link:** https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000

* This notebook implements the complete step-by-step pipeline for automated skin cancer classification from dermoscopic images, incorporating data leakage mitigation and class imbalance handling.


* Convolutional Neural Networks (CNNs) automate feature extraction and classification in medical imaging, achieving high accuracy for early disease detection.

* Diagnostic imaging is central to disease detection in radiology and dermatology. This problem asks students to build a Convolutional Neural Network (CNN) to classify disease presence directly from medical images, and compare it against a transfer-learning approach.


**Business Goal**


---



* The primary business objective of this project is to automate early detection and classification of skin cancer to assist dermatologists in clinical decision-making.

* By training a CNN model on this dataset, healthcare organizations aim to create a scalable, reliable, and rapid triage tool.

 **Executive Summary**


---
* **Clinical Problem**: Traditional skin cancer screening depends heavily on subjective human visual checks, creating diagnostic delays and a critical shortage of specialist care in remote regions.

* **Core Objective**: This project builds an automated deep learning pipeline using Convolutional Neural Networks (CNNs) to classify skin lesions into seven distinct diagnostic categories (dx) for rapid patient triage.

* **Data Engineering**: The workflow standardizes 10,015 high-resolution images from the HAM10000 dataset, employing lesion_id patient-group splitting to prevent data leakage and cheating during validation.

* **Modeling Strategy**: To counter the severe 67% benign class imbalance, the framework uses data augmentation and weighted loss functions while transitioning from a custom CNN baseline to advanced pre-trained architectures (ResNet/EfficientNet).

* **Clinical Impact**: Performance tracking prioritizes Sensitivity (Recall) over standard accuracy to prevent missing deadly malignancies, ultimately aiming to democratize early dermatological screening via scalable digital interfaces.



## Step 2: Dataset Details, Data Loading & Data Dictionary

**The Primary Visual Feature**


---



**The Image Data (image_id):** This is the core input for a Convolutional Neural Network (CNN). It consists of raw RGB pixel values from 450 × 600 color dermoscopic photographs. The model analyzes these pixels to extract visual patterns like color variation, border irregularity, asymmetry, and texture.

 **Key Clinical & Demographic Features**



---


These attributes from the metadata provide vital diagnostic context and help prevent demographic bias during model training:

**localization:** Specifies the body site where the lesion is located (e.g., back, face, lower extremity). This is clinically important because certain skin cancers are significantly more common on sun-exposed skin compared to protected areas.

**age:** The age of the patient at the time of examination. Age is a strong risk factor in dermatology; malignant tumors (like melanoma or basal cell carcinoma) are statistically much more common in older populations, while certain benign moles are prevalent in younger individuals.

**sex:**The biological sex of the patient (male, female, or unknown). This helps the model account for statistical differences in skin cancer prevalence and common anatomical locations between genders.

**dx_type (Ground Truth Method):** Describes how the diagnosis was confirmed (e.g., histopathology, expert consensus, or long-term follow-up). While not an input feature for prediction, it is a critical validation feature. Biopsy/histopathology represents the highest-quality "gold standard" ground truth for training.

**Structural Identifiers**



---



**lesion_id:** A tracking ID for the specific skin lesion. This is a critical structural feature for data splitting.  

Because a single lesion can have multiple photos taken from different angles, you must group your train/test splits by lesion_id to prevent data leakage (ensuring photos of the same mole do not accidentally end up in both training and testing sets).

In [11]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight

import kagglehub


dataset_dir = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Path to dataset directory:", dataset_dir)

# Locate the metadata CSV file inside the downloaded directory
metadata_path = os.path.join(dataset_dir, 'HAM10000_metadata.csv')

# Load the tabular metadata
df = pd.read_csv(metadata_path)
print(f"Dataset loaded successfully. Total rows: {len(df)}")
df.head()

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Path to dataset directory: /kaggle/input/skin-cancer-mnist-ham10000
Dataset loaded successfully. Total rows: 10015


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [6]:
# 1. General Info and Summary Statistics
print("--- Dataset Info ---")
df.info()

print("\n--- Descriptive Statistics for Numeric Features ---")
display(df.describe(include='all'))

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10015 entries, 0 to 10014
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   lesion_id     10015 non-null  object 
 1   image_id      10015 non-null  object 
 2   dx            10015 non-null  object 
 3   dx_type       10015 non-null  object 
 4   age           9958 non-null   float64
 5   sex           10015 non-null  object 
 6   localization  10015 non-null  object 
dtypes: float64(1), object(6)
memory usage: 547.8+ KB

--- Descriptive Statistics for Numeric Features ---


,lesion_id,image_id,dx,dx_type,age,sex,localization
count,10015,10015,10015,10015,9958.000000,10015,10015
unique,7470,10015,7,4,NaN,3,15
top,HAM_0000835,ISIC_0032258,nv,histo,NaN,male,back
freq,6,1,6705,5340,NaN,5406,2192
mean,NaN,NaN,NaN,NaN,51.863828,NaN,NaN
std,NaN,NaN,NaN,NaN,16.968614,NaN,NaN
min,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,40.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,50.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,65.000000,NaN,NaN


### Dataset Feature Dictionary

| Feature | Type | Description |
| :--- | :--- | :--- |
| **lesion_id** | String (Categorical) | Unique identifier for each lesion (important for tracking multiple photos of the same skin lesion to prevent data leakage) |
| **image_id** | String (Categorical) | Unique identifier for each individual dermoscopic image |
| **dx** | String (Categorical) | The diagnostic category (target label). The 7 classes are: `actnic keratoses (akiec)`, `basal cell carcinoma (bcc)`, `benign keratosis-like lesions (bkl)`, `dermatofibroma (df)`, `melanoma (mel)`, `melanocytic nevi (nv)`, and `vascular lesions (vasc)` |
| **dx_type** | String (Categorical) | The method of confirmation for the diagnosis: `histo` (histopathology), `consensus` (expert consensus), `confocal` (reflectance confocal microscopy), or `follow_up` (clinical follow-up) |
| **age** | Float (Numerical) | Age of the patient |
| **sex** | String (Categorical) | Gender of the patient (`male`, `female`, or `unknown`) |
| **localization** | String (Categorical) | Anatomical site of the lesion (e.g., `back`, `face`, `chest`, `abdomen`, etc.) |

In [9]:
# # 2. Missing Value Analysis
# print("--- Missing Values Per Column ---")
# missing_values = df.isnull().sum()
# display(missing_values[missing_values > 0])

In [10]:
# # 3. Target Class (dx) Distribution and Visualizations
# plt.figure(figsize=(15, 5))

# # Class Balance
# plt.subplot(1, 3, 1)
# sns.countplot(data=df, x='dx', order=df['dx'].value_counts().index, palette='viridis')
# plt.title('Distribution of Diagnosis (dx)')
# plt.xlabel('Diagnosis')
# plt.ylabel('Count')

# # Gender Distribution
# plt.subplot(1, 3, 2)
# sns.countplot(data=df, x='sex', order=df['sex'].value_counts().index, palette='muted')
# plt.title('Gender Distribution')
# plt.xlabel('Sex')
# plt.ylabel('Count')

# # Age Distribution
# plt.subplot(1, 3, 3)
# sns.histplot(data=df, x='age', bins=20, kde=True, color='skyblue')
# plt.title('Age Distribution')
# plt.xlabel('Age')
# plt.ylabel('Frequency')

# plt.tight_layout()
# plt.show()